# ABC2Vec — Notebook 09: Cross-Tradition Transfer

Can ABC2Vec — trained primarily on Irish folk — generalize to other folk traditions?

**Experiments:**
1. Zero-shot retrieval on BFDB (British folk) and Nottingham
2. Embedding quality on out-of-domain tunes
3. Cross-tradition similarity analysis — do British and Irish tunes cluster separately?
4. Transfer fine-tuning: does a small fine-tune on BFDB improve performance?

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score, normalized_mutual_info_score

try:
    import umap
except ImportError:
    !pip install umap-learn
    import umap

try:
    import faiss
except ImportError:
    !pip install faiss-cpu
    import faiss

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
BENCHMARK_DIR = PROJECT_DIR / 'data' / 'benchmark'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

## 9.1 Load Model & Cross-Tradition Data

In [ ]:
from abc2vec.model import ABC2VecModel, ABC2VecConfig
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier

# Load vocab + model
vocab = ABCVocabulary()
vocab_path = PROJECT_DIR / 'data' / 'processed' / 'vocabulary.json'
if vocab_path.exists():
    vocab.load(vocab_path)

patchifier = BarPatchifier(vocab, max_bar_length=64, max_bars=64)

config = ABC2VecConfig(
    vocab_size=len(vocab),
    max_bar_len=64, max_bars=64,
    d_model=256, n_heads=8, n_layers=6,
    d_ff=1024, d_embed=128, dropout=0.1,
)
model = ABC2VecModel(config).to(device)

ckpt_path = PROJECT_DIR / 'checkpoints' / 'best_model.pt'
if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint")
model.eval()

# Load cross-tradition data
cross_path = BENCHMARK_DIR / 'cross_tradition_test.parquet'
if cross_path.exists():
    cross_df = pd.read_parquet(cross_path)
    print(f"Cross-tradition data: {len(cross_df)} tunes")
    print(f"Traditions: {cross_df['tradition'].value_counts().to_dict()}")
else:
    # Load from processed data directly
    cross_entries = []
    for name, tradition in [('bfdb.json', 'british'), ('nottingham.json', 'nottingham')]:
        path = PROCESSED_DIR / name
        if path.exists():
            with open(path) as f:
                data = json.load(f)
            for e in data:
                e['tradition'] = tradition
                cross_entries.append(e)
    cross_df = pd.DataFrame(cross_entries)
    print(f"Loaded {len(cross_df)} cross-tradition tunes")

# Also load some Irish test data for comparison
irish_test_path = PROCESSED_DIR / 'test.parquet'
if irish_test_path.exists():
    irish_test = pd.read_parquet(irish_test_path).head(2000)
    irish_test['tradition'] = 'irish'
    print(f"Irish test: {len(irish_test)} tunes")

## 9.2 Encode All Traditions

In [ ]:
@torch.no_grad()
def encode_tunes(abc_texts, model, patchifier, device, batch_size=64):
    model.eval()
    all_emb = []
    for i in tqdm(range(0, len(abc_texts), batch_size), desc="Encoding"):
        batch = abc_texts[i:i+batch_size]
        bar_ids_list, bar_mask_list = [], []
        for text in batch:
            bars = patchifier.patchify(text)
            ids, mask = patchifier.encode(bars)
            bar_ids_list.append(ids)
            bar_mask_list.append(mask)
        bar_ids = torch.stack(bar_ids_list).to(device)
        bar_mask = torch.stack(bar_mask_list).to(device)
        emb = model.get_embedding(bar_ids, bar_mask)
        all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, axis=0)

# Encode each tradition
print("Encoding cross-tradition tunes...")
cross_emb = encode_tunes(cross_df['abc_body'].tolist(), model, patchifier, device)

print("\nEncoding Irish test tunes...")
irish_emb = encode_tunes(irish_test['abc_body'].tolist(), model, patchifier, device)

# Combine for visualization
all_emb = np.concatenate([irish_emb, cross_emb], axis=0)
all_traditions = np.array(
    ['irish'] * len(irish_emb) + cross_df['tradition'].tolist()
)
print(f"\nTotal: {len(all_emb)} embeddings")

## 9.3 Cross-Tradition UMAP Visualization

In [ ]:
# UMAP colored by tradition
print("Computing UMAP...")
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                     metric='cosine', random_state=42)
umap_2d = reducer.fit_transform(all_emb)

fig, ax = plt.subplots(figsize=(12, 10))

tradition_colors = {'irish': '#2ecc71', 'british': '#e74c3c', 'nottingham': '#3498db'}

for tradition, color in tradition_colors.items():
    mask = all_traditions == tradition
    if mask.sum() > 0:
        ax.scatter(umap_2d[mask, 0], umap_2d[mask, 1],
                   c=color, label=f"{tradition} (n={mask.sum()})",
                   s=8, alpha=0.5)

ax.set_title('UMAP — ABC2Vec Embeddings Colored by Tradition', fontsize=14)
ax.legend(markerscale=3, fontsize=12)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cross_tradition_umap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9.4 Tradition Separability Analysis

In [ ]:
# Can we predict tradition from embedding?
# High accuracy = embeddings capture tradition-specific features
# Low accuracy = embeddings are tradition-agnostic (more universal)

le_trad = LabelEncoder()
y_trad = le_trad.fit_transform(all_traditions)

clf_trad = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial', solver='lbfgs')
cv_trad = cross_val_score(clf_trad, all_emb, y_trad, cv=5, scoring='accuracy')

print(f"Tradition Classification (linear probe):")
print(f"  Accuracy: {cv_trad.mean():.4f} ± {cv_trad.std():.4f}")
print(f"  Classes: {list(le_trad.classes_)}")
print(f"\n  Interpretation:")
print(f"    High accuracy → embeddings are tradition-specific")
print(f"    Low accuracy → embeddings are more universal/transferable")

# Silhouette by tradition
sil_trad = silhouette_score(all_emb, y_trad)
nmi_trad = normalized_mutual_info_score(y_trad, 
    __import__('sklearn.cluster', fromlist=['KMeans']).KMeans(
        n_clusters=len(le_trad.classes_), random_state=42, n_init=10
    ).fit_predict(all_emb)
)
print(f"\n  Silhouette by tradition: {sil_trad:.4f}")
print(f"  NMI by tradition: {nmi_trad:.4f}")

## 9.5 Cross-Tradition Nearest Neighbors

In [ ]:
# For each British/Nottingham tune, find nearest Irish tune
# This reveals cross-tradition melodic connections

# Build FAISS index on Irish tunes
irish_norms = np.linalg.norm(irish_emb, axis=1, keepdims=True)
irish_norms = np.maximum(irish_norms, 1e-8)
irish_norm = (irish_emb / irish_norms).astype(np.float32)

index = faiss.IndexFlatIP(irish_emb.shape[1])
index.add(irish_norm)

# Query with cross-tradition tunes
cross_norms = np.linalg.norm(cross_emb, axis=1, keepdims=True)
cross_norms = np.maximum(cross_norms, 1e-8)
cross_norm = (cross_emb / cross_norms).astype(np.float32)

distances, nn_indices = index.search(cross_norm, 5)

# Show examples
print("Cross-Tradition Nearest Neighbors:")
print("=" * 60)

np.random.seed(42)
sample_idx = np.random.choice(len(cross_df), min(10, len(cross_df)), replace=False)

for idx in sample_idx:
    cross_tune = cross_df.iloc[idx]
    print(f"\n{cross_tune.get('tradition', 'unknown').upper()} tune: "
          f"{cross_tune.get('title', cross_tune.get('tune_name', 'N/A'))}")
    print(f"  ABC: {str(cross_tune.get('abc_body', ''))[:80]}...")
    
    print(f"  Nearest Irish tunes:")
    for rank in range(3):
        i_idx = nn_indices[idx, rank]
        irish_tune = irish_test.iloc[i_idx]
        print(f"    {rank+1}. {irish_tune.get('title', 'N/A')} "
              f"(sim={distances[idx, rank]:.4f})")

## 9.6 Cross-Tradition Tune Type Classification

In [ ]:
# Can a classifier trained on Irish tune types generalize to British tunes?
VALID_TYPES = ['reel', 'jig', 'hornpipe', 'polka', 'waltz', 'march']

# Train on Irish
irish_mask = irish_test['tune_type'].isin(VALID_TYPES)
X_train = irish_emb[irish_mask.values]
y_train = irish_test.loc[irish_mask, 'tune_type'].values

clf = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial', solver='lbfgs')
clf.fit(X_train, y_train)

# Evaluate on cross-tradition (if tune_type is available)
if 'tune_type' in cross_df.columns:
    cross_mask = cross_df['tune_type'].isin(VALID_TYPES)
    if cross_mask.sum() > 0:
        X_test = cross_emb[cross_mask.values]
        y_test = cross_df.loc[cross_mask, 'tune_type'].values
        
        y_pred = clf.predict(X_test)
        acc = np.mean(y_pred == y_test)
        
        print(f"Cross-tradition tune type transfer:")
        print(f"  Train: Irish ({len(X_train)} samples)")
        print(f"  Test: {len(X_test)} cross-tradition samples")
        print(f"  Zero-shot accuracy: {acc:.4f}")
    else:
        print("No cross-tradition tunes with matching tune types.")
else:
    # Use predictions as analysis
    print("No tune_type labels in cross-tradition data.")
    print("Predicting types for cross-tradition tunes:")
    pred_types = clf.predict(cross_emb)
    from collections import Counter
    print(f"  Predicted type distribution: {dict(Counter(pred_types))}")

## 9.7 Embedding Quality: Cosine Similarity Distributions

In [ ]:
# Compare pairwise similarity distributions within/across traditions
from sklearn.metrics.pairwise import cosine_similarity

# Sample for efficiency
n_sample = min(500, len(irish_emb), len(cross_emb))
np.random.seed(42)
irish_sample = irish_emb[np.random.choice(len(irish_emb), n_sample, replace=False)]
cross_sample = cross_emb[np.random.choice(len(cross_emb), n_sample, replace=False)]

# Within-Irish similarities
sim_irish = cosine_similarity(irish_sample)
triu_idx = np.triu_indices(n_sample, k=1)
irish_sims = sim_irish[triu_idx]

# Within-cross similarities
sim_cross = cosine_similarity(cross_sample)
cross_sims = sim_cross[triu_idx]

# Cross-tradition similarities
sim_across = cosine_similarity(irish_sample, cross_sample)
across_sims = sim_across.flatten()
# Subsample for plotting
across_sims = np.random.choice(across_sims, min(len(irish_sims), len(across_sims)), replace=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(irish_sims, bins=80, alpha=0.5, label='Within Irish', color='green', density=True)
ax.hist(cross_sims, bins=80, alpha=0.5, label='Within British/Nottingham', color='red', density=True)
ax.hist(across_sims, bins=80, alpha=0.5, label='Irish ↔ British', color='blue', density=True)
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('Pairwise Cosine Similarity Distributions by Tradition')
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cross_tradition_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Within Irish:    mean={irish_sims.mean():.4f}, std={irish_sims.std():.4f}")
print(f"Within Cross:    mean={cross_sims.mean():.4f}, std={cross_sims.std():.4f}")
print(f"Across:          mean={across_sims.mean():.4f}, std={across_sims.std():.4f}")

## 9.8 Summary

In [ ]:
print("=" * 60)
print("Cross-Tradition Transfer — Summary")
print("=" * 60)
print(f"\nTraining tradition: Irish (IrishMAN)")
print(f"Test traditions: British (BFDB), Nottingham")
print(f"\nKey findings:")
print(f"  1. Tradition separability (probe acc): {cv_trad.mean():.4f}")
print(f"  2. Silhouette by tradition: {sil_trad:.4f}")
print(f"  3. Within-Irish sim: {irish_sims.mean():.4f}")
print(f"  4. Cross-tradition sim: {across_sims.mean():.4f}")
print(f"\nInterpretation:")
print(f"  - Similar within/across similarity → good transfer")
print(f"  - High tradition probe accuracy → tradition-specific features captured")
print(f"  - Low silhouette → traditions overlap in embedding space (universal features)")